In [3]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=100,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    random_state=42
)

model = GradientBoostingClassifierScratch(n_estimators=20)
model.fit(X, y)

preds = model.predict(X)
print(preds[:10])

[0 1 0 0 1 0 0 1 0 1]


In [1]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

class GradientBoostingClassifierScratch:
    
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=2):
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.models = []
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def fit(self, X, y):
        # Step 1: initialize log-odds
        pos = np.sum(y)
        neg = len(y) - pos
        self.F0 = np.log(pos / neg)
        
        F = np.full(len(y), self.F0)
        
        for _ in range(self.n_estimators):
            # Step 2: probabilities
            p = self.sigmoid(F)
            
            # Step 3: residuals (gradient)
            residuals = y - p
            
            # Step 4: fit weak learner
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            
            # Step 5: update
            F += self.lr * tree.predict(X)
            
            self.models.append(tree)
    
    def predict_proba(self, X):
        F = np.full(X.shape[0], self.F0)
        
        for tree in self.models:
            F += self.lr * tree.predict(X)
        
        return self.sigmoid(F)
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return (probs >= 0.5).astype(int)